# Model Operationalization

## Deployment Environmets
The model should be deployed on the company's internal servers due to the confidential nature of weekly sales data. Aditionally, large-scaled cloud computing resources are not required, as the computational demands of the model are relatively low.

## Operations

### Develop ARIMA and SARIMA production models

In [1]:
import time
import mlflow
import pickle
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

walmart_minimal = pd.read_parquet("../data/processed/walmart_minimal.parquet")
arima_stores_id = [4, 7, 14, 16, 18, 30, 33, 36, 38, 42, 44]
arima_stores = (
    walmart_minimal[
        walmart_minimal["unique_id"].isin(arima_stores_id)
    ]
    .sort_values(by=["unique_id", "ds"])
)
sarima_stores = (
    walmart_minimal[
        ~walmart_minimal["unique_id"].isin(arima_stores_id)
    ]
    .sort_values(by=["unique_id", "ds"])
)

c:\ProgramData\miniconda3\envs\ml_walmart\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("sqlite:///../mlflow/mlflow.db")
experiment_name = "walmart_models_operationalization"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location="../mlflow/mlruns/"
    )
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_id=experiment_id)

2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/14 20:43:26 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/14 20:43:26 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/14 20:43:26 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<Experiment: artifact_location='file:///c:/Users/olver/Desktop/ml_walmart/notebooks/../mlflow/mlruns', creation_time=1771112606919, experiment_id='2', last_update_time=1771112606919, lifecycle_stage='active', name='walmart_models_operationalization', tags={}>

In [8]:
mae_list = []
rmse_list = []

shared_runtime = {
    "freq": "W",
    "n_jobs": -1
}

shared_cv = {
    "h": 12,
    "n_windows": 3,
}
configs = [
    {
        "input": arima_stores,
        "input_name": "arima_stores",
        "model": AutoARIMA(alias="ARIMA"),
        "name": "ARIMA",
        "model_params": None
    },
    {
        "input": sarima_stores,
        "input_name": "sarima_stores",
        "model": AutoARIMA(alias="SARIMA", season_length=52),
        "name": "SARIMA",
        "model_params": {
            "season_length": 52
        }
    }
]
def make_cross_validation(config, sf: StatsForecast, name: str):
    cv = sf.cross_validation(
        df=config["input"],
        **shared_cv
    )
    run_id = mlflow.active_run().info.run_id
    cv_path = f"../artifacts/cv_predictions/{name}_{run_id}_operationalization.csv"
    cv.to_csv(cv_path, index = False)
    mlflow.log_artifact(cv_path)
    return cv

def evaluate_cross_validation(cv: pd.DataFrame, name: str):
    eval_df = evaluate(cv, metrics=[mae, rmse])
    run_id = mlflow.active_run().info.run_id
    eval_path = f"../artifacts/metrics/{name}_{run_id}_operationalization.csv"
    eval_df.to_csv(eval_path, index=False)
    mlflow.log_artifact(eval_path)
    return eval_df

def log_evaluation_metrics(config, eval_df: pd.DataFrame, name: str):
    mae_df = eval_df[eval_df["metric"] == "mae"]
    rmse_df = eval_df[eval_df["metric"] == "rmse"]
    mae_val = mae_df[name].mean()
    rmse_val = rmse_df[name].mean()
    mae_list.append(mae_val)
    rmse_list.append(rmse_val)
    mlflow.log_metric("mae", mae_val)
    mlflow.log_metric("rmse", rmse_val)
    for _, row in mae_df.iterrows():
        store_id = row["unique_id"]
        mlflow.log_metric(f"mae_store_{store_id}", row[name])
    for _, row in rmse_df.iterrows():
        store_id = row["unique_id"]
        mlflow.log_metric(f"rmse_store_{store_id}", row[name])
    print(config["name"], f"mae: {mae_val}", f"rmse: {rmse_val}")

def fit_and_save_model(config, sf:StatsForecast, name: str):
    sf.fit(config["input"])
    run_id = mlflow.active_run().info.run_id
    model_path = f"../models/{name}_{run_id}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(sf, f)
    mlflow.log_artifact(model_path)

for config in configs:
    with mlflow.start_run(run_name=config["name"]):
        start = time.time()
        mlflow.log_params({
            "model": config["name"],
            **(config["model_params"] or {}),
            "dataset": config["input_name"],
            **shared_runtime,
            **shared_cv
        })
        sf = StatsForecast(
            models=[config["model"]],
            **shared_runtime
        )
        cv = make_cross_validation(config, sf, config["name"])
        eval_df = evaluate_cross_validation(cv, config["name"])
        log_evaluation_metrics(config, eval_df,config["name"])
        fit_and_save_model(config, sf, config["name"])

        elapsed = time.time() - start
        mlflow.log_metric("fit_time_sec", elapsed)
print(
    "ARIMA and SARIMA average",
    f"mae: {sum(mae_list)/len(mae_list)}",
    f"rmse: {sum(rmse_list)/len(rmse_list)}"
)

ARIMA mae: 31019.869041415644 rmse: 39167.178690886445
SARIMA mae: 34384.53798622836 rmse: 43949.76836170366
ARIMA and SARIMA average mae: 32702.203513822 rmse: 41558.473526295056
